# Chapter 18: Color

Source span: *Fundamentals of Computer Graphics*, Chapter 18, printed pages 503-524, physical PDF pages 520-541.

Chapter goal: turn color from an apparent property of RGB triples into a measured pipeline: spectrum -> observer integrals -> tristimulus coordinates -> color-space transforms -> display encoding -> perceptual and viewing-condition checks.

Color is the first topic in the book where the geometry is not mostly a shape in world space. The geometry is a set of coordinate systems and projections: spectra live in a high-dimensional function space, XYZ is a three-dimensional measurement space, chromaticity diagrams are projective 2D normalizations, RGB gamuts are triangles or volumes carved out by physical primaries, and L*a*b* / LMS spaces model perceptual or physiological axes. The notebook keeps those spaces separate and records checks at every conversion boundary.

## Translation guide

| Source idea | Computational object used here | What can go wrong |
| --- | --- | --- |
| Spectrum Phi(lambda) | sampled power array over visible wavelengths | treating a spectrum as if it were already RGB |
| Cone/color matching response | numerical integral of spectrum times observer functions | expecting one wavelength detector instead of three broad integrators |
| Grassmann laws | linearity tests on spectra and tristimulus values | doing nonlinear display operations before color mixing |
| CIE XYZ | device-independent 3-vector | assuming every XYZ value is physically displayable |
| xy and u'v' chromaticity | normalized projections of XYZ | reading distances in xy as perceptual distances |
| RGB device space | matrix from primaries plus white point | losing the color space tag or mixing encoded values |
| sRGB transfer curve | piecewise encode/decode functions | averaging or lighting in gamma-encoded values |
| L*a*b* and Delta E | opponent-space transform relative to a white | treating L*a*b* as a chromaticity diagram |
| Chromatic adaptation | LMS diagonal gain transform | clipping or quantizing before white balance |

The source span gives the conceptual route: colorimetry, color spaces, chromatic adaptation, and color appearance. The code below uses small synthetic spectra and patches so every result is reproducible without copying textbook figures or relying on external image assets.

## Visual storyboard

| Order | Visual artifact | Concept | Representation and library | Learner inspection target | Validation |
| --- | --- | --- | --- | --- | --- |
| 1 | `spectra-tristimulus-metamerism.png` | spectra vs tristimulus values, color matching, metamerism | sampled spectra with NumPy integrals and Matplotlib | two visibly different spectra can have the same XYZ response | metamer residual and Grassmann linearity error |
| 2 | `chromaticity-gamut-geometry.png` | xy/u'v' projection and gamut triangles | CIE-like locus, sRGB triangle, Matplotlib geometry | sRGB is a triangle inside a broader spectral locus; u'v' changes spacing | triangle area, white-point-in-triangle, out-of-gamut spectral wavelengths |
| 3 | `rgb-xyz-transform-display-checks.png` | constructing RGB<->XYZ matrices from primaries | matrix construction plus swatches in Matplotlib | primaries and white point determine the transform, not the letters R/G/B | matrix inverse and D65 white reconstruction |
| 4 | `transfer-function-quantization-lab.png` and `.html` | sRGB nonlinear transfer and quantization | Matplotlib static lab plus Plotly HTML | display code values are not linear light; encoded mixing is too dark | monotonic encode/decode, round-trip residual, quantization step comparison |
| 5 | `lab-deltae-perceptual-pitfalls.png` | approximate perceptual distances | L*a*b* transform and Delta E bars | equal RGB steps do not mean equal perceptual steps | L*a*b* round trip and positive Delta E ordering |
| 6 | `von-kries-adaptation-white-balance.png` | chromatic adaptation and white balancing | LMS diagonal gain model | white can be mapped across illuminants without rerendering, but clipping and 8-bit quantization matter | white residual, adaptation round trip, quantization amplification summary |

Library routing: NumPy handles sampled spectra and matrix algebra; Matplotlib is the right fit for durable 2D diagrams, swatches, chromaticity geometry, and static audits; Plotly is used only where a transfer-function lab benefits from trace toggling; SciPy is used only for the convex-hull area check behind gamut geometry.

In [ ]:
from pathlib import Path
import sys

CHAPTER = 18
TOPIC = "chapter-18"
TITLE = "Color"
PRINTED_PAGES = "503-524"
PDF_PAGES = "520-541"

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for folder in [ARTIFACT_ROOT / "figures", ARTIFACT_ROOT / "html", ARTIFACT_ROOT / "checks", ARTIFACT_ROOT / "tables"]:
    folder.mkdir(parents=True, exist_ok=True)

ARTIFACT_ROOT.relative_to(BOOK_ROOT)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import patches
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.spatial import ConvexHull

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib, save_plotly_html, save_table_csv
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, close, style_axis

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 9})

In [ ]:
# Core colorimetry and color-space utilities used throughout the chapter.
WAVELENGTHS = np.arange(380.0, 781.0, 1.0)


def _piecewise_gaussian(wavelengths, center, left_scale, right_scale, amplitude=1.0):
    wavelengths = np.asarray(wavelengths, dtype=float)
    scale = np.where(wavelengths < center, left_scale, right_scale)
    return amplitude * np.exp(-0.5 * ((wavelengths - center) * scale) ** 2)


def cie_1931_xyz_fit(wavelengths):
    # Analytic approximation to the CIE 1931 2-degree color matching functions.
    wavelengths = np.asarray(wavelengths, dtype=float)
    x = (_piecewise_gaussian(wavelengths, 442.0, 0.0624, 0.0374, 0.362)
         + _piecewise_gaussian(wavelengths, 599.8, 0.0264, 0.0323, 1.056)
         - _piecewise_gaussian(wavelengths, 501.1, 0.0490, 0.0382, 0.065))
    y = (_piecewise_gaussian(wavelengths, 568.8, 0.0213, 0.0247, 0.821)
         + _piecewise_gaussian(wavelengths, 530.9, 0.0613, 0.0322, 0.286))
    z = (_piecewise_gaussian(wavelengths, 437.0, 0.0845, 0.0278, 1.217)
         + _piecewise_gaussian(wavelengths, 459.0, 0.0385, 0.0725, 0.681))
    return np.clip(np.stack([x, y, z], axis=-1), 0.0, None)


CMF = cie_1931_xyz_fit(WAVELENGTHS)
TRAPZ_WEIGHTS = np.ones_like(WAVELENGTHS)
TRAPZ_WEIGHTS[[0, -1]] = 0.5
TRAPZ_WEIGHTS *= WAVELENGTHS[1] - WAVELENGTHS[0]
CMF_INTEGRATION = CMF.T * TRAPZ_WEIGHTS

SRGB_PRIMARIES_XY = np.array([[0.6400, 0.3300], [0.3000, 0.6000], [0.1500, 0.0600]])
D65_XY = np.array([0.3127, 0.3290])
ILLUMINANT_A_XY = np.array([0.44757, 0.40745])
HPE_XYZ_TO_LMS = np.array([[0.38971, 0.68898, -0.07868], [-0.22981, 1.18340, 0.04641], [0.0, 0.0, 1.0]])
HPE_LMS_TO_XYZ = np.linalg.inv(HPE_XYZ_TO_LMS)


def gaussian_spectrum(center, width, amplitude=1.0):
    return amplitude * np.exp(-0.5 * ((WAVELENGTHS - center) / width) ** 2)


def xyz_from_spectrum(power):
    return CMF_INTEGRATION @ np.asarray(power, dtype=float)


def xy_from_xyz(xyz):
    xyz = np.asarray(xyz, dtype=float)
    denom = np.sum(xyz, axis=-1, keepdims=True)
    return xyz[..., :2] / np.maximum(denom, 1e-12)


def uv_from_xyz(xyz):
    xyz = np.asarray(xyz, dtype=float)
    denom = xyz[..., 0:1] + 15.0 * xyz[..., 1:2] + 3.0 * xyz[..., 2:3]
    return np.concatenate([4.0 * xyz[..., 0:1], 9.0 * xyz[..., 1:2]], axis=-1) / np.maximum(denom, 1e-12)


def uv_from_xy(xy):
    xy = np.asarray(xy, dtype=float)
    denom = -2.0 * xy[..., 0:1] + 12.0 * xy[..., 1:2] + 3.0
    return np.concatenate([4.0 * xy[..., 0:1], 9.0 * xy[..., 1:2]], axis=-1) / np.maximum(denom, 1e-12)


def xyz_from_xyY(xy, Y=1.0):
    x, y = np.asarray(xy, dtype=float)
    return np.array([x * Y / y, Y, (1.0 - x - y) * Y / y])


def rgb_to_xyz_matrix_from_primaries(primaries_xy, white_xy):
    primary_xyz = np.stack([xyz_from_xyY(xy, 1.0) for xy in primaries_xy], axis=1)
    white_xyz = xyz_from_xyY(white_xy, 1.0)
    scales = np.linalg.solve(primary_xyz, white_xyz)
    return primary_xyz * scales


SRGB_TO_XYZ = rgb_to_xyz_matrix_from_primaries(SRGB_PRIMARIES_XY, D65_XY)
XYZ_TO_SRGB = np.linalg.inv(SRGB_TO_XYZ)
D65_XYZ = xyz_from_xyY(D65_XY, 1.0)
ILLUMINANT_A_XYZ = xyz_from_xyY(ILLUMINANT_A_XY, 1.0)


def srgb_encode(linear):
    linear = np.clip(np.asarray(linear, dtype=float), 0.0, 1.0)
    return np.where(linear <= 0.0031308, 12.92 * linear, 1.055 * np.power(linear, 1.0 / 2.4) - 0.055)


def srgb_decode(encoded):
    encoded = np.clip(np.asarray(encoded, dtype=float), 0.0, 1.0)
    return np.where(encoded <= 0.04045, encoded / 12.92, np.power((encoded + 0.055) / 1.055, 2.4))


def rgb_to_xyz(linear_rgb):
    return np.asarray(linear_rgb, dtype=float) @ SRGB_TO_XYZ.T


def xyz_to_rgb(xyz):
    return np.asarray(xyz, dtype=float) @ XYZ_TO_SRGB.T


def xyz_for_display(xyz, target_Y=0.72):
    xyz = np.asarray(xyz, dtype=float)
    return xyz * (target_Y / max(float(xyz[..., 1].max()), 1e-12))


def display_rgb_from_xyz(xyz):
    return srgb_encode(np.clip(xyz_to_rgb(xyz), 0.0, 1.0))


def shoelace_area(points):
    points = np.asarray(points, dtype=float)
    x = points[:, 0]
    y = points[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def barycentric_coordinates(point, triangle):
    point = np.asarray(point, dtype=float)
    a, b, c = np.asarray(triangle, dtype=float)
    uv = np.linalg.solve(np.column_stack((a - c, b - c)), point - c)
    return np.array([uv[0], uv[1], 1.0 - uv.sum()])


def point_in_triangle(point, triangle, tol=1e-10):
    bary = barycentric_coordinates(point, triangle)
    return bool(np.all(bary >= -tol)), bary


def lab_f(t):
    t = np.asarray(t, dtype=float)
    delta = 6.0 / 29.0
    return np.where(t > delta**3, np.cbrt(t), t / (3.0 * delta**2) + 4.0 / 29.0)


def lab_f_inv(ft):
    ft = np.asarray(ft, dtype=float)
    delta = 6.0 / 29.0
    return np.where(ft > delta, ft**3, 3.0 * delta**2 * (ft - 4.0 / 29.0))


def xyz_to_lab(xyz, white=D65_XYZ):
    ratio = np.asarray(xyz, dtype=float) / np.asarray(white, dtype=float)
    f = lab_f(ratio)
    return np.stack([116.0 * f[..., 1] - 16.0, 500.0 * (f[..., 0] - f[..., 1]), 200.0 * (f[..., 1] - f[..., 2])], axis=-1)


def lab_to_xyz(lab, white=D65_XYZ):
    lab = np.asarray(lab, dtype=float)
    fy = (lab[..., 0] + 16.0) / 116.0
    fx = fy + lab[..., 1] / 500.0
    fz = fy - lab[..., 2] / 200.0
    return np.stack([lab_f_inv(fx), lab_f_inv(fy), lab_f_inv(fz)], axis=-1) * np.asarray(white, dtype=float)


def delta_e_ab(lab_a, lab_b):
    return np.linalg.norm(np.asarray(lab_a) - np.asarray(lab_b), axis=-1)


def von_kries_matrix(source_white_xyz, target_white_xyz):
    source_lms = HPE_XYZ_TO_LMS @ source_white_xyz
    target_lms = HPE_XYZ_TO_LMS @ target_white_xyz
    return HPE_LMS_TO_XYZ @ np.diag(target_lms / source_lms) @ HPE_XYZ_TO_LMS

computed_srgb_reference = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
assert np.allclose(SRGB_TO_XYZ, computed_srgb_reference, atol=8e-4)
assert np.allclose(SRGB_TO_XYZ @ XYZ_TO_SRGB, np.eye(3), atol=1e-12)


## Spectra, Integrals, And Metamers

A spectrum is a sampled function of wavelength. XYZ is not a resampling of that function; it is three weighted integrals against observer functions. That single change explains why color devices can use three channels and also why the three channels are not enough to recover the original spectrum.

The figure below builds two spectra that are deliberately different but have the same XYZ values under the approximate CIE observer. The construction uses a smooth null direction: a wavelength-dependent perturbation whose integral against x_bar, y_bar, and z_bar is zero. Adding or subtracting that perturbation changes the spectrum while leaving the tristimulus response unchanged. This is the computational form of metamerism. The same cell checks Grassmann-style linearity by verifying that integration commutes with positive spectral mixtures.

In [ ]:
base_spectrum = 0.9 + 0.12 * np.sin((WAVELENGTHS - 380.0) / 400.0 * 2.0 * np.pi)
raw_null = gaussian_spectrum(445.0, 22.0) - 0.8 * gaussian_spectrum(535.0, 30.0) + 0.55 * gaussian_spectrum(635.0, 36.0)
projection = CMF_INTEGRATION.T @ np.linalg.solve(CMF_INTEGRATION @ CMF_INTEGRATION.T, CMF_INTEGRATION @ raw_null)
metamer_null = raw_null - projection
alpha = 0.85 * base_spectrum.min() / np.max(np.abs(metamer_null))
spectrum_a = base_spectrum + alpha * metamer_null
spectrum_b = base_spectrum - alpha * metamer_null
xyz_a = xyz_from_spectrum(spectrum_a)
xyz_b = xyz_from_spectrum(spectrum_b)
metamer_relative_residual = float(np.linalg.norm(xyz_a - xyz_b) / np.linalg.norm((xyz_a + xyz_b) / 2.0))
grassmann_linearity_error = float(np.linalg.norm(xyz_from_spectrum(0.35 * spectrum_a + 1.7 * spectrum_b) - (0.35 * xyz_a + 1.7 * xyz_b)))
swatch_rgb = display_rgb_from_xyz(xyz_for_display(np.stack([xyz_a, xyz_b]), target_Y=0.65))

fig, axes = plt.subplots(2, 2, figsize=(11, 7), gridspec_kw={"height_ratios": [2.2, 1.0]})
ax = axes[0, 0]
ax.plot(WAVELENGTHS, spectrum_a, color=PALETTE["blue"], label="spectrum A")
ax.plot(WAVELENGTHS, spectrum_b, color=PALETTE["red"], label="spectrum B")
ax.fill_between(WAVELENGTHS, spectrum_a, spectrum_b, color="#8aa1b6", alpha=0.16, label="spectral difference")
style_axis(ax, "Different spectra", xlabel="wavelength (nm)", ylabel="relative power")
ax.legend(loc="upper right")
ax = axes[0, 1]
ax.plot(WAVELENGTHS, CMF[:, 0], color="#c44e52", label="x_bar fit")
ax.plot(WAVELENGTHS, CMF[:, 1], color="#6a994e", label="y_bar fit")
ax.plot(WAVELENGTHS, CMF[:, 2], color="#2f6fbb", label="z_bar fit")
style_axis(ax, "Observer weighting functions", xlabel="wavelength (nm)", ylabel="relative sensitivity")
ax.legend(loc="upper right")
ax = axes[1, 0]
x = np.arange(3)
ax.bar(x - 0.17, xyz_a / xyz_a[1], 0.34, color=PALETTE["blue"], label="A normalized by Y")
ax.bar(x + 0.17, xyz_b / xyz_b[1], 0.34, color=PALETTE["red"], alpha=0.78, label="B normalized by Y")
ax.set_xticks(x, ["X", "Y", "Z"])
style_axis(ax, "Same tristimulus response", ylabel="relative value")
ax.text(0.02, 0.90, f"relative residual {metamer_relative_residual:.2e}", transform=ax.transAxes, fontsize=8)
ax.legend(loc="upper right")
ax = axes[1, 1]
ax.set_axis_off()
for idx, (label, rgb) in enumerate([("A", swatch_rgb[0]), ("B", swatch_rgb[1])]):
    ax.add_patch(patches.Rectangle((0.08 + idx * 0.44, 0.22), 0.34, 0.48, facecolor=rgb, edgecolor="#2b2f33"))
    ax.text(0.25 + idx * 0.44, 0.12, f"display swatch {label}", ha="center")
ax.text(0.5, 0.82, "swatches match because XYZ matches", ha="center", fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.suptitle("Spectra become colors only after observer integration", y=0.995, fontsize=13)
fig.tight_layout()
spectra_path = save_matplotlib(fig, TOPIC, "spectra-tristimulus-metamerism.png")
close(fig)
display_artifact(spectra_path, width=860)
spectra_checks = {"metamer_relative_residual": metamer_relative_residual, "grassmann_linearity_error": grassmann_linearity_error, "spectrum_a_min": float(spectrum_a.min()), "spectrum_b_min": float(spectrum_b.min())}
spectra_checks

## Chromaticity And Gamut Geometry

The xy diagram is a projection: divide XYZ by X + Y + Z and keep two coordinates. That makes the visible locus easy to sketch and turns a three-primary display into a triangle, but it also throws away luminance and distorts perceptual distances. The u'v' normalization keeps the same source measurements but spreads parts of the diagram differently.

Do not color the whole horseshoe as if a monitor can display it. A physical RGB display can only make additive mixtures inside its primary triangle. The spectral locus is drawn as a boundary and the sRGB/BT.709 triangle is drawn as the reproducible device gamut used by the later matrix and transfer-function checks.

In [ ]:
locus_mask = CMF.sum(axis=1) > 1e-5
locus_xyz = CMF[locus_mask]
locus_wavelengths = WAVELENGTHS[locus_mask]
locus_xy = xy_from_xyz(locus_xyz)
locus_uv = uv_from_xyz(locus_xyz)
srgb_triangle_xy = np.vstack([SRGB_PRIMARIES_XY, SRGB_PRIMARIES_XY[0]])
srgb_triangle_uv = np.vstack([uv_from_xy(SRGB_PRIMARIES_XY), uv_from_xy(SRGB_PRIMARIES_XY[0])])
d65_uv = uv_from_xy(D65_XY)
inside_d65_xy, d65_bary = point_in_triangle(D65_XY, SRGB_PRIMARIES_XY)
xy_hull = ConvexHull(locus_xy)
uv_hull = ConvexHull(locus_uv)
spectral_hull_area_xy = float(xy_hull.volume)
spectral_hull_area_uv = float(uv_hull.volume)
srgb_area_xy = float(shoelace_area(SRGB_PRIMARIES_XY))
srgb_area_uv = float(shoelace_area(uv_from_xy(SRGB_PRIMARIES_XY)))
locus_unitY = locus_xyz / np.maximum(locus_xyz[:, 1:2], 1e-12)
locus_rgb = xyz_to_rgb(locus_unitY)
out_of_srgb_count = int(np.count_nonzero(np.any((locus_rgb < -1e-8) | (locus_rgb > 1.0 + 1e-8), axis=1)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, points, triangle, white, title, xlabel, ylabel in [(axes[0], locus_xy, srgb_triangle_xy, D65_XY, "CIE xy chromaticity", "x", "y"), (axes[1], locus_uv, srgb_triangle_uv, d65_uv, "CIE u'v' chromaticity", "u'", "v'")]:
    ax.plot(points[:, 0], points[:, 1], color=PALETTE["ink"], linewidth=1.6, label="spectral locus fit")
    ax.plot([points[0, 0], points[-1, 0]], [points[0, 1], points[-1, 1]], color=PALETTE["violet"], linewidth=1.2, linestyle="--", label="purple boundary")
    ax.plot(triangle[:, 0], triangle[:, 1], color=PALETTE["red"], linewidth=2.0, label="sRGB / BT.709 gamut")
    ax.scatter(white[0], white[1], s=48, color=PALETTE["gold"], edgecolor="#2b2f33", zorder=4, label="D65 white")
    for wl in [460, 520, 560, 610, 700]:
        idx = int(np.argmin(np.abs(locus_wavelengths - wl)))
        ax.text(points[idx, 0], points[idx, 1], f"{wl}", fontsize=8, ha="center", va="bottom")
    style_axis(ax, title, equal=True, xlabel=xlabel, ylabel=ylabel)
    ax.legend(loc="upper right")
axes[0].text(0.05, 0.06, f"sRGB xy area / locus hull area = {srgb_area_xy / spectral_hull_area_xy:.2f}", transform=axes[0].transAxes, fontsize=8)
axes[1].text(0.05, 0.06, "same spectra, different normalization", transform=axes[1].transAxes, fontsize=8)
fig.suptitle("Chromaticity diagrams expose gamut as geometry", y=0.995, fontsize=13)
fig.tight_layout()
chromaticity_path = save_matplotlib(fig, TOPIC, "chromaticity-gamut-geometry.png")
close(fig)
display_artifact(chromaticity_path, width=900)
chromaticity_checks = {"inside_d65_srgb_triangle": inside_d65_xy, "d65_barycentric_coordinates": [float(v) for v in d65_bary], "srgb_area_xy": srgb_area_xy, "spectral_hull_area_xy": spectral_hull_area_xy, "srgb_area_uv": srgb_area_uv, "spectral_hull_area_uv": spectral_hull_area_uv, "out_of_srgb_spectral_wavelength_count": out_of_srgb_count, "sampled_locus_points": int(locus_xy.shape[0])}
chromaticity_checks

## Constructing RGB To XYZ From Primaries

The source span constructs an RGB transform from four measured chromaticities: red, green, blue, and the white point. The code does the same thing for the BT.709/sRGB primaries. It first turns each (x, y) into an XYZ direction with Y = 1, then solves for three scale factors so that RGB = (1, 1, 1) lands on the display white.

This matters operationally: RGB numbers are not self-describing. The letters R, G, and B only become a color space after the primaries, white point, matrix, and transfer curve are known.

In [ ]:
unit_rgb = np.eye(3)
primary_xyz = rgb_to_xyz(unit_rgb)
primary_xy = xy_from_xyz(primary_xyz)
white_from_matrix = rgb_to_xyz(np.ones(3))
white_xy_from_matrix = xy_from_xyz(white_from_matrix)
source_matrix_inverse_residual = float(np.linalg.norm(SRGB_TO_XYZ @ XYZ_TO_SRGB - np.eye(3)))
white_xy_error = float(np.linalg.norm(white_xy_from_matrix - D65_XY))
sample_linear_rgb = np.array([[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,.8,.05],[.05,.35,1],[.4,.4,.4]], dtype=float)
sample_xyz = rgb_to_xyz(sample_linear_rgb)
rgb_xyz_roundtrip_error = float(np.max(np.abs(xyz_to_rgb(sample_xyz) - sample_linear_rgb)))
matrix_rows = [{"row": row_label, "R": row[0], "G": row[1], "B": row[2]} for row_label, row in zip(["X", "Y", "Z"], SRGB_TO_XYZ)]
matrix_table_path = save_table_csv(matrix_rows, TOPIC, "srgb-rgb-to-xyz-matrix.csv")

fig, axes = plt.subplots(1, 3, figsize=(12, 4.4), gridspec_kw={"width_ratios": [1.25, 1.0, 1.2]})
ax = axes[0]
im = ax.imshow(SRGB_TO_XYZ, cmap="viridis", vmin=0, vmax=max(0.95, SRGB_TO_XYZ.max()))
ax.set_xticks([0,1,2], ["R","G","B"]); ax.set_yticks([0,1,2], ["X","Y","Z"])
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{SRGB_TO_XYZ[i,j]:.4f}", ha="center", va="center", color="white" if SRGB_TO_XYZ[i,j] > 0.45 else "black", fontsize=8)
ax.set_title("sRGB / BT.709 RGB -> XYZ")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax = axes[1]
ax.plot(srgb_triangle_xy[:,0], srgb_triangle_xy[:,1], color=PALETTE["red"], linewidth=2)
ax.scatter(primary_xy[:,0], primary_xy[:,1], c=["#e34a33", "#31a354", "#3182bd"], s=70, edgecolor="#222")
for name, xy in zip(["red", "green", "blue"], primary_xy):
    ax.text(xy[0]+0.01, xy[1]+0.01, name, fontsize=8)
ax.scatter(white_xy_from_matrix[0], white_xy_from_matrix[1], c=PALETTE["gold"], edgecolor="#222", s=62, label="RGB=(1,1,1)")
style_axis(ax, "primaries plus white", equal=True, xlabel="x", ylabel="y")
ax.legend(loc="upper right")
ax = axes[2]
ax.set_axis_off()
for i, rgb in enumerate(sample_linear_rgb):
    y = 0.86 - i * 0.12
    ax.add_patch(patches.Rectangle((0.04, y - 0.04), 0.30, 0.075, facecolor=srgb_encode(rgb), edgecolor="#222", linewidth=0.8))
    xy = xy_from_xyz(sample_xyz[i])
    ax.text(0.40, y, f"linear RGB {rgb.round(2).tolist()} -> xy=({xy[0]:.3f}, {xy[1]:.3f})", va="center", fontsize=8)
ax.text(0.04, 0.96, "sample display colors after sRGB encoding", fontsize=9, weight="bold")
ax.set_xlim(0,1); ax.set_ylim(0,1)
fig.suptitle("RGB coordinates need primaries, a white point, and a matrix", y=0.995, fontsize=13)
fig.tight_layout()
rgb_matrix_path = save_matplotlib(fig, TOPIC, "rgb-xyz-transform-display-checks.png")
close(fig)
display_artifact(rgb_matrix_path, width=900)
display_artifact(matrix_table_path)
rgb_matrix_checks = {"matrix_inverse_residual": source_matrix_inverse_residual, "white_xy_error": white_xy_error, "rgb_xyz_roundtrip_error": rgb_xyz_roundtrip_error, "white_xyz_from_rgb_111": [float(v) for v in white_from_matrix], "matrix_table": matrix_table_path.relative_to(BOOK_ROOT).as_posix()}
rgb_matrix_checks

## Transfer Function Lab: Linear Light Is Not Display Code

The sRGB standard uses the same primaries as BT.709 but adds a nonlinear transfer function. The nonlinearity is not a lighting model. It is an encoding choice that allocates more code values where the visual system is more sensitive and where displays historically had nonlinear response.

The lab below compares three operations: encoding linear light to sRGB for storage or display, decoding sRGB before arithmetic, and averaging two display-encoded colors directly. The static PNG is audit-friendly; the HTML artifact lets a learner toggle curves and inspect code values.

In [ ]:
linear_samples = np.linspace(0.0, 1.0, 2048)
encoded_samples = srgb_encode(linear_samples)
decoded_samples = srgb_decode(encoded_samples)
pure_gamma_22 = np.power(linear_samples, 1.0 / 2.2)
transfer_roundtrip_error = float(np.max(np.abs(decoded_samples - linear_samples)))
transfer_monotonic = bool(np.all(np.diff(encoded_samples) >= 0.0))
threshold = 0.0031308
transfer_toe_continuity_error = float(abs(12.92 * threshold - (1.055 * threshold ** (1.0 / 2.4) - 0.055)))
codes = np.linspace(0.0, 1.0, 256)
linear_from_srgb_codes = srgb_decode(codes)
srgb_linear_steps = np.diff(linear_from_srgb_codes)
quantization_ratio = float(srgb_linear_steps[-1] / srgb_linear_steps[0])
red_srgb = np.array([1.0, 0.0, 0.0]); green_srgb = np.array([0.0, 1.0, 0.0])
wrong_encoded_average = 0.5 * (red_srgb + green_srgb)
correct_linear_average = srgb_encode(0.5 * (srgb_decode(red_srgb) + srgb_decode(green_srgb)))
encoded_mix_luminance_ratio = float(rgb_to_xyz(srgb_decode(wrong_encoded_average))[1] / rgb_to_xyz(srgb_decode(correct_linear_average))[1])

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
ax = axes[0,0]
ax.plot(linear_samples, encoded_samples, color=PALETTE["blue"], label="sRGB encode")
ax.plot(linear_samples, pure_gamma_22, color=PALETTE["gray"], linestyle="--", label="power 1/2.2")
ax.plot(linear_samples, linear_samples, color=PALETTE["ink"], linewidth=1, alpha=.65, label="identity")
ax.axvline(threshold, color=PALETTE["gold"], linestyle=":", linewidth=1.2, label="sRGB toe")
style_axis(ax, "encoding curve", xlabel="linear light", ylabel="stored/display code")
ax.legend(loc="lower right")
ax = axes[0,1]
ax.plot(codes[1:], srgb_linear_steps, color=PALETTE["red"], label="decoded sRGB step")
ax.axhline(1/255, color=PALETTE["ink"], linestyle="--", linewidth=1.1, label="uniform linear 8-bit step")
style_axis(ax, "quantization after decode", xlabel="sRGB code value", ylabel="linear step size")
ax.legend(loc="upper left")
ax = axes[1,0]
ax.set_axis_off()
for i, (label, rgb) in enumerate([("red", red_srgb), ("green", green_srgb), ("wrong: average encoded", wrong_encoded_average), ("correct: average linear, encode", correct_linear_average)]):
    y = 0.82 - i * 0.20
    ax.add_patch(patches.Rectangle((0.08, y - 0.055), 0.30, 0.11, facecolor=np.clip(rgb, 0, 1), edgecolor="#222"))
    ax.text(0.44, y, label, va="center", fontsize=9)
ax.text(0.08, 0.96, "encoded arithmetic changes brightness", fontsize=10, weight="bold")
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax = axes[1,1]
ramp = np.tile(np.linspace(0, 1, 512), (34, 1))
ramps = np.vstack([np.dstack([srgb_encode(ramp)] * 3), np.ones((8,512,3)), np.dstack([ramp] * 3), np.ones((8,512,3)), np.dstack([srgb_encode(srgb_decode(ramp))] * 3)])
ax.imshow(ramps, aspect="auto", extent=[0,1,0,1])
ax.set_yticks([.82,.50,.18], ["linear encoded", "linear sent as code", "decode+encode"])
ax.set_xticks([0,.5,1])
ax.set_title("display ramp sanity check")
for spine in ax.spines.values(): spine.set_visible(False)
fig.suptitle("sRGB transfer functions protect display values, not scene-linear math", y=0.995, fontsize=13)
fig.tight_layout()
transfer_path = save_matplotlib(fig, TOPIC, "transfer-function-quantization-lab.png")
close(fig)
plotly_fig = go.Figure()
plotly_fig.add_trace(go.Scatter(x=linear_samples, y=encoded_samples, mode="lines", name="sRGB encode"))
plotly_fig.add_trace(go.Scatter(x=linear_samples, y=pure_gamma_22, mode="lines", name="power 1/2.2", line={"dash":"dash"}))
plotly_fig.add_trace(go.Scatter(x=linear_samples, y=linear_samples, mode="lines", name="identity", line={"dash":"dot"}))
plotly_fig.add_trace(go.Scatter(x=codes[1:], y=srgb_linear_steps / srgb_linear_steps.max(), mode="lines", name="normalized decoded step size", yaxis="y2"))
plotly_fig.update_layout(title="sRGB transfer function lab", xaxis_title="linear light or sRGB code value", yaxis_title="encoded value", yaxis2={"title":"normalized quantization step", "overlaying":"y", "side":"right", "range":[0,1.05]}, template="plotly_white", legend={"orientation":"h", "y":-0.22}, height=520)
transfer_html_path = save_plotly_html(plotly_fig, TOPIC, "transfer-function-quantization-lab.html")
display_artifact(transfer_path, width=860)
display_artifact(transfer_html_path, width="100%", height=520)
transfer_checks = {"transfer_roundtrip_error": transfer_roundtrip_error, "transfer_monotonic": transfer_monotonic, "toe_continuity_error": transfer_toe_continuity_error, "last_to_first_decoded_step_ratio": quantization_ratio, "encoded_mix_luminance_ratio_wrong_over_correct": encoded_mix_luminance_ratio}
transfer_checks

## L*a*b*: Useful Distances, Not A Chromaticity Diagram

L*a*b* adds an explicit white reference and separates lightness from two opponent axes. It is useful for approximate color differences, but the a* and b* values still depend on luminance through the nonlinear transform. The plot below uses equal-size steps in encoded RGB and shows that their Delta E values are not equal.

This is the perceptual warning behind many rendering and texture bugs: small numerical differences in one color space may be large or small differences to the observer depending on lightness, hue, and viewing context.

In [ ]:
encoded_samples_for_lab = np.array([[.10,.10,.10],[.45,.45,.45],[.85,.85,.85],[.82,.22,.18],[.18,.70,.25],[.18,.28,.86]])
encoded_samples_step = np.clip(encoded_samples_for_lab + np.array([.045,.045,.045]), 0, 1)
linear_lab_samples = srgb_decode(encoded_samples_for_lab)
linear_lab_step_samples = srgb_decode(encoded_samples_step)
lab_a = xyz_to_lab(rgb_to_xyz(linear_lab_samples))
lab_b = xyz_to_lab(rgb_to_xyz(linear_lab_step_samples))
delta_e_values = delta_e_ab(lab_a, lab_b)
lab_roundtrip_error = float(np.max(np.abs(lab_to_xyz(lab_a) - rgb_to_xyz(linear_lab_samples))))
delta_rows = [{"sample": i, "encoded_R": rgb[0], "encoded_G": rgb[1], "encoded_B": rgb[2], "L_star": lab[0], "a_star": lab[1], "b_star": lab[2], "delta_E_for_equal_rgb_step": de} for i, (rgb, lab, de) in enumerate(zip(encoded_samples_for_lab, lab_a, delta_e_values), start=1)]
delta_table_path = save_table_csv(delta_rows, TOPIC, "lab-deltae-sample-patches.csv")
fig, axes = plt.subplots(1, 3, figsize=(12, 4.3), gridspec_kw={"width_ratios":[1.1,1.25,1.05]})
ax = axes[0]
for i, (rgb, lab) in enumerate(zip(encoded_samples_for_lab, lab_a), start=1):
    ax.scatter(lab[1], lab[2], s=92, color=rgb, edgecolor="#222", zorder=3)
    ax.text(lab[1]+2, lab[2]+2, str(i), fontsize=8)
ax.axhline(0, color="#c6ccd2", linewidth=.8); ax.axvline(0, color="#c6ccd2", linewidth=.8)
style_axis(ax, "opponent axes", equal=True, xlabel="a* green <- -> red", ylabel="b* blue <- -> yellow")
ax = axes[1]
pos = np.arange(len(delta_e_values))
ax.bar(pos, delta_e_values, color=[tuple(rgb) for rgb in encoded_samples_for_lab], edgecolor="#222")
ax.set_xticks(pos, [str(i) for i in range(1, len(delta_e_values)+1)])
style_axis(ax, "same encoded RGB step, different Delta E", xlabel="sample patch", ylabel="Delta E*ab")
ax.text(.03,.93, f"max/min = {delta_e_values.max()/delta_e_values.min():.2f}", transform=ax.transAxes, fontsize=8)
ax = axes[2]
ax.set_axis_off()
for i, (rgb0, rgb1) in enumerate(zip(encoded_samples_for_lab, encoded_samples_step)):
    y = .86 - i * .14
    ax.add_patch(patches.Rectangle((.08,y-.045), .22, .08, facecolor=rgb0, edgecolor="#222"))
    ax.add_patch(patches.Rectangle((.34,y-.045), .22, .08, facecolor=rgb1, edgecolor="#222"))
    ax.text(.62, y, f"patch {i+1}: Delta E {delta_e_values[i]:.2f}", va="center", fontsize=8)
ax.text(.08,.96, "before and after equal encoded step", fontsize=9, weight="bold")
ax.set_xlim(0,1); ax.set_ylim(0,1)
fig.suptitle("L*a*b* exposes perceptual distance pitfalls in RGB editing", y=0.995, fontsize=13)
fig.tight_layout()
lab_path = save_matplotlib(fig, TOPIC, "lab-deltae-perceptual-pitfalls.png")
close(fig)
display_artifact(lab_path, width=900)
display_artifact(delta_table_path)
lab_checks = {"lab_roundtrip_xyz_max_error": lab_roundtrip_error, "delta_e_min": float(delta_e_values.min()), "delta_e_max": float(delta_e_values.max()), "delta_e_max_to_min_ratio": float(delta_e_values.max()/delta_e_values.min()), "delta_table": delta_table_path.relative_to(BOOK_ROOT).as_posix()}
lab_checks

## Chromatic Adaptation And Display Checks

Chromatic adaptation is modeled here with the source span's key simplification: transform XYZ into an LMS cone-response space, scale each channel independently, and transform back. The model is a white-balance operation, not a new shading calculation. It maps a source adapting white to a target adapting white and applies the same gain logic to nearby colors.

The display check is deliberately practical. The warm-illuminant column is converted directly to an sRGB display, so it may clip or look too warm. The white-balanced column applies the inverse adaptation back to D65. The numeric checks record whether white maps correctly, whether the round trip returns the original patches, and how much extra error appears if values are quantized too early.

In [ ]:
d65_to_A = von_kries_matrix(D65_XYZ, ILLUMINANT_A_XYZ)
A_to_d65 = von_kries_matrix(ILLUMINANT_A_XYZ, D65_XYZ)
white_mapping_error = float(np.linalg.norm(d65_to_A @ D65_XYZ - ILLUMINANT_A_XYZ))
adaptation_identity_error = float(np.linalg.norm(A_to_d65 @ d65_to_A - np.eye(3)))
reference_linear_rgb = np.array([[1,1,1],[1,.78,.05],[.20,.55,.92],[.18,.72,.30],[.72,.28,.76]], dtype=float)
reference_xyz = rgb_to_xyz(reference_linear_rgb)
warm_xyz = reference_xyz @ d65_to_A.T
balanced_xyz = warm_xyz @ A_to_d65.T
balanced_linear_rgb = xyz_to_rgb(balanced_xyz)
warm_linear_rgb = xyz_to_rgb(warm_xyz)
reference_roundtrip_error = float(np.max(np.abs(balanced_linear_rgb - reference_linear_rgb)))
warm_out_of_gamut_count = int(np.count_nonzero(np.any((warm_linear_rgb < 0) | (warm_linear_rgb > 1), axis=1)))
warm_quantized_rgb = np.round(np.clip(warm_linear_rgb, 0, 1) * 255) / 255
balanced_from_quantized_rgb = xyz_to_rgb(rgb_to_xyz(warm_quantized_rgb) @ A_to_d65.T)
quantization_amplification_error = float(np.max(np.abs(np.clip(balanced_from_quantized_rgb, 0, 1) - reference_linear_rgb)))
fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.6))
columns = [("reference under D65", reference_linear_rgb), ("warm illuminant A, direct display", np.clip(warm_linear_rgb,0,1)), ("after von Kries white balance", np.clip(balanced_linear_rgb,0,1))]
for ax, (title, linear_rgbs) in zip(axes, columns):
    ax.set_axis_off(); ax.set_title(title, fontsize=10)
    for i, rgb in enumerate(linear_rgbs):
        y = .84 - i * .16
        ax.add_patch(patches.Rectangle((.16,y-.055), .68, .10, facecolor=srgb_encode(rgb), edgecolor="#222", linewidth=.9))
        ax.text(.08, y, str(i+1), va="center", ha="center", fontsize=8)
    ax.set_xlim(0,1); ax.set_ylim(0,1)
axes[1].text(.5,.04, f"out-of-gamut patches before clipping: {warm_out_of_gamut_count}", ha="center", fontsize=8)
axes[2].text(.5,.04, f"round-trip max RGB error: {reference_roundtrip_error:.2e}", ha="center", fontsize=8)
fig.suptitle("Chromatic adaptation as LMS gain control and display white balance", y=0.995, fontsize=13)
fig.tight_layout()
adaptation_path = save_matplotlib(fig, TOPIC, "von-kries-adaptation-white-balance.png")
close(fig)
display_artifact(adaptation_path, width=880)
adaptation_checks = {"white_mapping_error": white_mapping_error, "adaptation_identity_error": adaptation_identity_error, "reference_roundtrip_linear_rgb_error": reference_roundtrip_error, "warm_out_of_srgb_gamut_count": warm_out_of_gamut_count, "quantization_before_adaptation_max_rgb_error": quantization_amplification_error}
adaptation_checks

## Applied lab

Use this lab as a small color-management debugging routine.

1. Pick one patch in `reference_linear_rgb` and change one channel by a small amount. Re-run the L*a*b* cell and compare RGB distance to Delta E. If the Delta E change is much larger than expected, the edit is perceptually visible even if the RGB delta looks small.
2. Change `ILLUMINANT_A_XY` to another white point, such as a cooler illuminant near `(0.28, 0.30)`. Re-run the adaptation cell. The white-mapping error should stay near zero, but the out-of-gamut count and quantization error can change.
3. In the transfer lab, replace the correct linear average with encoded averaging for another pair of colors. Predict whether the result becomes too dark or too bright before looking at the swatch.

A good lab answer includes both a visual observation and a failed or changed invariant. For example: the blue patch clipped after warm adaptation and `warm_out_of_srgb_gamut_count` increased, or the swatch looked similar but Delta E doubled.

## Sanity checks

These checks are the notebook's compact color-management contract:

- observer integration is linear and can map different spectra to the same tristimulus values,
- chromaticity projection reconstructs XYZ when paired with Y,
- the RGB-to-XYZ matrix is invertible and maps `(1, 1, 1)` to the D65 white point,
- sRGB encode/decode is monotone and round-trips linear values,
- L*a*b* round-trips through XYZ for the tested patches,
- von Kries adaptation maps adapting whites and reverses cleanly in floating point,
- every concept artifact exists and is nonblank.

The JSON artifacts are intentionally part of the lesson: they state what each visual is supposed to prove, so later notebook edits can be audited without re-reading the prose.

In [ ]:
xy_reconstruct_source = np.array([0.41, 0.36])
xyz_reconstructed = xyz_from_xyY(xy_reconstruct_source, 0.72)
xy_reconstruction_error = float(np.linalg.norm(xy_from_xyz(xyz_reconstructed) - xy_reconstruct_source))
all_checks = {"chapter": CHAPTER, "title": TITLE, "source_span": {"printed_pages": PRINTED_PAGES, "pdf_pages": PDF_PAGES}, "spectra": spectra_checks, "chromaticity": chromaticity_checks, "rgb_xyz_matrix": rgb_matrix_checks, "transfer_function": transfer_checks, "lab_deltae": lab_checks, "chromatic_adaptation": adaptation_checks, "xy_reconstruction_error": xy_reconstruction_error}
assert spectra_checks["metamer_relative_residual"] < 1e-11
assert spectra_checks["grassmann_linearity_error"] < 1e-10
assert chromaticity_checks["inside_d65_srgb_triangle"]
assert chromaticity_checks["srgb_area_xy"] < chromaticity_checks["spectral_hull_area_xy"]
assert chromaticity_checks["out_of_srgb_spectral_wavelength_count"] > 0
assert rgb_matrix_checks["matrix_inverse_residual"] < 1e-12
assert rgb_matrix_checks["white_xy_error"] < 1e-12
assert rgb_matrix_checks["rgb_xyz_roundtrip_error"] < 1e-12
assert transfer_checks["transfer_monotonic"]
assert transfer_checks["transfer_roundtrip_error"] < 1e-12
assert transfer_checks["toe_continuity_error"] < 5e-8
assert transfer_checks["encoded_mix_luminance_ratio_wrong_over_correct"] < 0.55
assert lab_checks["lab_roundtrip_xyz_max_error"] < 1e-12
assert lab_checks["delta_e_max_to_min_ratio"] > 1.25
assert adaptation_checks["white_mapping_error"] < 1e-12
assert adaptation_checks["adaptation_identity_error"] < 1e-12
assert adaptation_checks["reference_roundtrip_linear_rgb_error"] < 1e-12
assert xy_reconstruction_error < 1e-12
checks_path = save_json(all_checks, TOPIC, "color-computation-checks.json")
display_artifact(checks_path)
all_checks

In [ ]:
created_artifacts = [spectra_path, chromaticity_path, rgb_matrix_path, transfer_path, transfer_html_path, lab_path, adaptation_path, matrix_table_path, delta_table_path, checks_path]
artifact_records = assert_artifacts(created_artifacts)
image_records = [assert_nonblank_image(path) for path in [spectra_path, chromaticity_path, rgb_matrix_path, transfer_path, lab_path, adaptation_path]]
artifact_sanity = {"chapter": CHAPTER, "artifact_count_before_sanity_file": len(created_artifacts), "artifacts": artifact_records, "nonblank_images": image_records, "checks_json": str(checks_path.relative_to(BOOK_ROOT))}
artifact_sanity_path = save_json(artifact_sanity, TOPIC, "artifact-sanity.json")
assert_artifacts([artifact_sanity_path])
display_artifact(artifact_sanity_path)
artifact_sanity

## Takeaways

- A color pipeline is a chain of spaces. Name the space before doing arithmetic.
- Spectral differences can disappear after three observer integrals; that is useful for displays and dangerous for spectral reconstruction.
- xy and u'v' diagrams are geometric projections. Gamut boundaries are real constraints, while diagram distances are not a universal perceptual metric.
- RGB matrices come from primaries and a white point. The same three numbers mean different colors under a different RGB definition.
- sRGB values are encoded display/storage values. Decode before lighting, averaging, filtering, or interpolation in linear light.
- L*a*b* and Delta E help reason about perceptual differences, but they still depend on the reference white and do not replace color appearance modeling.
- White balance can be modeled as an LMS gain change, but it should be applied before clipping and low-bit-depth quantization whenever possible.